In [3]:
import os, warnings, time
import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
start_time = time.time()

solar_model_df         = pd.read_csv("solar_model_dataset.csv")
wind_onshore_model_df  = pd.read_csv("wind_onshore_model_dataset.csv")
wind_offshore_model_df = pd.read_csv("wind_offshore_model_dataset.csv")

for df in [solar_model_df, wind_onshore_model_df, wind_offshore_model_df]:
    df["datetime"] = pd.to_datetime(df["datetime"])

solar_model_df         = solar_model_df.sort_values("datetime").reset_index(drop=True)
wind_onshore_model_df  = wind_onshore_model_df.sort_values("datetime").reset_index(drop=True)
wind_offshore_model_df = wind_offshore_model_df.sort_values("datetime").reset_index(drop=True)

# print(solar_model_df.info())
# print(wind_onshore_model_df.info())
# print(wind_offshore_model_df.info())

solar_model_df[
    ["global_radiation",
     "sunshine_duration",
     "solar_zenith_angle",
     "air_temperature"]
] = (
    solar_model_df[
        ["global_radiation",
         "sunshine_duration",
         "solar_zenith_angle",
         "air_temperature"]
    ]
    .ffill()
)



DATASETS = {
    "Solar":         solar_model_df,
    "Wind_Onshore":  wind_onshore_model_df,
    "Wind_Offshore": wind_offshore_model_df
}

print("Datasets loaded:")
for name, df in DATASETS.items():
    print(f"  {name}: {df.shape[0]:,} rows | "
          f"{df['datetime'].min()} → {df['datetime'].max()}")

# ==========================================================
# 3. EXPERIMENT CONFIGURATION  (identical to Notebook 05)
# ==========================================================

PERIODS = {
    "FULL": {
        "start":     "2015-01-08 00:00:00",
        "train_end": "2022-12-31 23:00:00",
        "valid_end": "2023-12-31 23:00:00"
    },
    "POST_2020": {
        "start":     "2020-01-01 00:00:00",
        "train_end": "2023-12-31 23:00:00",
        "valid_end": "2024-12-31 23:00:00"
    },
    "POST_2021": {
        "start":     "2021-01-01 00:00:00",
        "train_end": "2023-12-31 23:00:00",
        "valid_end": "2024-12-31 23:00:00"
    }
}

TARGETS = {
    "Solar":         "generation_photovoltaics_mwh",
    "Wind_Onshore":  "generation_wind_onshore_mwh",
    "Wind_Offshore": "generation_wind_offshore_mwh"
}
HORIZONS = [1, 6, 24, 72, 168]


ARIMA_MODELS = ["SARIMA", "ARIMA", "SARIMAX"]

# ==========================================================
# 4. FIXED MODEL ORDERS
# Academically justified standard values used widely in
# renewable energy forecasting literature:
#   d=1  : first-order differencing (energy series non-stationary)
#   p=1,q=1: AR(1) + MA(1) — parsimonious, robust baseline
#   D=1,P=1,Q=1,s=24: seasonal differencing + AR + MA
#                      s=24 captures the daily cycle in hourly data
# ==========================================================

ARIMA_ORDER    = (1,1,1)
SEASONAL_ORDER = (1,1,1,24)
ARIMA_ONLY_SO  = (0, 0, 0, 0)    # ARIMA has no seasonal component

# ==========================================================
# 5. EXOGENOUS VARIABLES FOR SARIMAX
# From Notebook 03 EDA — highest correlated weather variables:
#   Solar        → global_radiation  (Pearson r = 0.885)
#   Wind Onshore → wind_speed        (Pearson r = 0.829)
#   Wind Offshore→ wind_speed        (Pearson r = 0.412)
# ==========================================================

# SARIMAX_EXOG = {
#     "Solar":         ["global_radiation"],
#     "Wind_Onshore":  ["wind_speed"],
#     "Wind_Offshore": ["wind_speed"]
# }

SARIMAX_EXOG = {
    "Solar": [
        "global_radiation",
        "sunshine_duration",
        "solar_zenith_angle",
        "air_temperature"
    ],
    "Wind_Onshore": [
    "wind_speed",
    "wind_dir_sin",
    "wind_dir_cos"

],
    "Wind_Offshore": [
    "wind_speed",
    "wind_dir_sin",
    "wind_dir_cos"

]
}



# ==========================================================
# 6. SPEED SETTINGS
# ==========================================================

TRAIN_WINDOW   = 90 * 24
REFIT_EVERY    = 720

# ==========================================================
# 7. RESULTS STORAGE
# ==========================================================

results = pd.DataFrame(columns=[
    "Period", "Target", "Horizon",
    "Model", "MAE", "RMSE", "MAPE", "R2"
])

# ==========================================================
# 8. UTILITY FUNCTIONS  (same as Notebook 05)
# ==========================================================

def calculate_mape(y_true, y_pred):
    y_true = np.where(y_true == 0, 1e-8, y_true)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def evaluate_model(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = calculate_mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return mae, rmse, mape, r2

def add_result(results, period, target, horizon, model,
               mae, rmse, mape, r2):
    results.loc[len(results)] = {
        "Period": period, "Target": target,
        "Horizon": horizon, "Model": model,
        "MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2
    }
    return results

# ==========================================================
# 9. DATA SPLITTING  (same as Notebook 05)
# ==========================================================

def get_period_split(df, period_name):
    p         = PERIODS[period_name]
    df_period = df[df["datetime"] >= p["start"]].copy()
    train_df  = df_period[df_period["datetime"] <= p["train_end"]].copy()
    valid_df  = df_period[
        (df_period["datetime"] >  p["train_end"]) &
        (df_period["datetime"] <= p["valid_end"])
    ].copy()
    return train_df, valid_df

# ==========================================================
# 10. WALK-FORWARD FORECAST
# - Uses last TRAIN_WINDOW rows from training set
# - Forecasts H steps at a time across validation set
# - Refits every REFIT_EVERY steps for freshness
# - Adds actual observed values to history after each block
# ==========================================================

def walk_forward_forecast(train_series, valid_series,
                          order, seasonal_order,
                          horizon,
                          train_exog=None, valid_exog=None):

    # Seed history from last TRAIN_WINDOW of training data
    history = list(
                    train_series.iloc[-TRAIN_WINDOW:].values
                    )
    exog_history = (
                    train_exog.iloc[-TRAIN_WINDOW:].values.tolist()
                    if train_exog is not None
                    else None
                    )
    valid_vals     = valid_series.values
    valid_exog_arr = valid_exog.values if valid_exog is not None else None
    n_valid        = len(valid_vals)
    predictions    = []

    # ── Initial fit ─────────────────────────────────────────
    try:

      mod    = SARIMAX(history,
                         exog=exog_history,
                         order=order,
                         seasonal_order=seasonal_order,
                         enforce_stationarity=False,
                         enforce_invertibility=False)
      fitted = mod.fit(disp=False, maxiter=50)

    except Exception as e:
        print(f"          Initial fit failed: {e}")
        return np.full(n_valid, np.nan)

    step = 0
    while step < n_valid:

        # ── Periodic refit ──────────────────────────────────
        if step > 0 and step % REFIT_EVERY == 0:
            try:
                mod = SARIMAX(
                        history[-TRAIN_WINDOW:],
                        exog=(
                                exog_history[-TRAIN_WINDOW:]
                                if exog_history is not None
                                else None
                                ),
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    )
                fitted = mod.fit(disp=False, maxiter=50)
            except Exception:
                pass   # keep previous model if refit fails

        # ── Forecast next H steps ───────────────────────────
        # FORECAST_BLOCK = max(horizon, 24)
        # steps_ahead = min(FORECAST_BLOCK, n_valid - step)
        steps_ahead = min(horizon, n_valid - step)
        try:
            fc_exog  = None
            if valid_exog_arr is not None:
                fc_exog = valid_exog_arr[step: step + steps_ahead]
            forecast = fitted.forecast(steps=steps_ahead, exog=fc_exog)
            preds    = np.array(forecast)
        except Exception:
            preds = np.full(steps_ahead, np.nan)

        predictions.extend(preds)

        # ── Update history with actual observed values ──────
        # ── Update history with actual observed values ──────
        new_y = valid_vals[step: step + steps_ahead]

        history.extend(new_y)

        if exog_history is not None:
            new_exog = valid_exog_arr[step: step + steps_ahead]

            exog_history.extend(new_exog.tolist())

            fitted = fitted.append(
                endog=new_y,
                exog=new_exog,
                refit=False
                    )

        else:
            fitted = fitted.append(
                endog=new_y,
                refit=False
                )

        # if step % 100 == 0:
            # print(step)

        step += steps_ahead

    return np.array(predictions[:n_valid])

# ==========================================================
# 11. ARIMA FAMILY RUNNER
# ==========================================================

def run_arima_model(model_name, target_col, target_h_col,
                    train_df, valid_df, target_name, horizon):

    train_series = train_df[target_col].copy()
    valid_series = valid_df[target_col].copy()
    y_true       = valid_df[target_h_col].dropna().values

    # Determine orders
    if model_name == "ARIMA":
        order         = ARIMA_ORDER
        seasonal_order= ARIMA_ONLY_SO
        train_exog    = None
        valid_exog    = None
    elif model_name == "SARIMA":
        order         = ARIMA_ORDER
        seasonal_order= SEASONAL_ORDER
        train_exog    = None
        valid_exog    = None
    else:  # SARIMAX
        order         = ARIMA_ORDER
        seasonal_order= SEASONAL_ORDER
        exog_cols     = SARIMAX_EXOG[target_name]
        train_exog    = train_df[exog_cols].copy()
        valid_exog    = valid_df[exog_cols].copy()

    # Walk-forward forecast
    predictions = walk_forward_forecast(
        train_series   = train_series,
        valid_series   = valid_series,
        order          = order,
        seasonal_order = seasonal_order,
        horizon        = horizon,
        train_exog     = train_exog,
        valid_exog     = valid_exog
    )

    # Align predictions with y_true
    if horizon == 1:

        preds_aligned = predictions

    else:
        forecast_positions = np.arange(
                                        horizon - 1,
                                        len(predictions),
                                        horizon
                                         )

        preds_aligned = predictions[forecast_positions]

    min_len        = min(len(preds_aligned), len(y_true))
    preds_aligned  = preds_aligned[:min_len]
    y_true_aligned = y_true[:min_len]

    # Remove NaNs
    mask           = ~np.isnan(preds_aligned)
    preds_aligned  = preds_aligned[mask]
    y_true_aligned = y_true_aligned[mask]

    if len(preds_aligned) == 0:
        return np.nan, np.nan, np.nan, np.nan

    # print("Prediction sample:")
    # print(preds_aligned[:20])

    # print("\nActual sample:")
    # print(y_true_aligned[:20])

    # print("\nPrediction mean:", np.mean(preds_aligned))
    # print("Actual mean:", np.mean(y_true_aligned))

    return evaluate_model(y_true_aligned, preds_aligned)

# ==========================================================
# 12. MASTER EXPERIMENT LOOP
# ==========================================================

TOTAL_RUNS = (
    len(PERIODS) * len(TARGETS) *
    len(HORIZONS) * len(ARIMA_MODELS)
)

print(f"\nTotal ARIMA Experiments : {TOTAL_RUNS}")
print(f"Models                  : {ARIMA_MODELS}")
print(f"Fixed ARIMA order       : {ARIMA_ORDER}")
print(f"Fixed seasonal order    : {SEASONAL_ORDER}  (s=24 daily cycle)")
print(f"Training window         : {TRAIN_WINDOW} hours ({TRAIN_WINDOW//24} days)")
print(f"Refit every             : {REFIT_EVERY} hours ({REFIT_EVERY//24} days)")
# print(f"Estimated runtime       : ~25–40 minutes")
print("="*60)

os.makedirs("results", exist_ok=True)
counter = 1

for period_name in PERIODS:
    for target_name in TARGETS:

        dataset    = DATASETS[target_name]
        target_col = TARGETS[target_name]

        for horizon in HORIZONS:
            for model_name in ARIMA_MODELS:

                exp_start = time.time()
                print(f"\n[{counter}/{TOTAL_RUNS}] "
                      f"{period_name} | {target_name} | "
                      f"H{horizon} | {model_name}",
                      end=" ... ", flush=True)
                counter += 1

                try:
                    train_df, valid_df = get_period_split(dataset, period_name)
                    target_h_col       = f"target_h{horizon}"

                    mae, rmse, mape, r2 = run_arima_model(
                        model_name   = model_name,
                        target_col   = target_col,
                        target_h_col = target_h_col,
                        train_df     = train_df,
                        valid_df     = valid_df,
                        target_name  = target_name,
                        horizon      = horizon
                    )

                    elapsed_exp = time.time() - exp_start
                    print(f"MAE={mae:.0f}  RMSE={rmse:.0f}  "
                          f"R²={r2:.3f}  [{elapsed_exp:.0f}s]")

                    results = add_result(
                        results, period_name, target_name,
                        horizon, model_name,
                        mae, rmse, mape, r2
                    )

                except Exception as e:
                    print(f"FAILED — {e}")

                # Save after every run — safe if interrupted
                results.to_csv("results/arima_results.csv", index=False)

# ==========================================================
# 13. SUMMARY
# ==========================================================

elapsed = (time.time() - start_time) / 60
print("\n" + "="*60)
print("ARIMA EXPERIMENTS COMPLETE")
print("="*60)
print(f"Completed : {len(results)}/{TOTAL_RUNS} experiments")
print(f"Runtime   : {elapsed:.1f} minutes")
print(f"Saved     : results/arima_results.csv")

print("\nOverall Model Ranking (avg RMSE):")
print(results.groupby("Model")[["RMSE","MAE","R2"]]
      .mean().sort_values("RMSE").round(2))

print("\nRMSE by Horizon:")
print(results.groupby("Horizon")["RMSE"].mean().sort_values().round(1))

print("\nRMSE by Target:")
print(results.groupby("Target")["RMSE"].mean().sort_values().round(1))

print("\nAll Results:")
print(results.to_string())

Datasets loaded:
  Solar: 98,244 rows | 2015-01-08 00:00:00 → 2026-03-24 22:00:00
  Wind_Onshore: 98,244 rows | 2015-01-08 00:00:00 → 2026-03-24 22:00:00
  Wind_Offshore: 98,244 rows | 2015-01-08 00:00:00 → 2026-03-24 22:00:00

Total ARIMA Experiments : 135
Models                  : ['SARIMA', 'ARIMA', 'SARIMAX']
Fixed ARIMA order       : (1, 1, 1)
Fixed seasonal order    : (1, 1, 1, 24)  (s=24 daily cycle)
Training window         : 2160 hours (90 days)
Refit every             : 720 hours (30 days)

[1/135] FULL | Solar | H1 | SARIMA ... MAE=1745  RMSE=2980  R²=0.908  [4805s]

[2/135] FULL | Solar | H1 | ARIMA ... MAE=2116  RMSE=3485  R²=0.874  [169s]

[3/135] FULL | Solar | H1 | SARIMAX ... MAE=1755  RMSE=2963  R²=0.909  [5169s]

[4/135] FULL | Solar | H6 | SARIMA ... MAE=7924  RMSE=12151  R²=-5.114  [990s]

[5/135] FULL | Solar | H6 | ARIMA ... MAE=10390  RMSE=15518  R²=-8.972  [33s]

[6/135] FULL | Solar | H6 | SARIMAX ... MAE=7791  RMSE=12058  R²=-5.021  [1361s]

[7/135] FULL | Sol